In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import uuid

# ==========================================================
# Configuration
# ==========================================================
bronze_table = "bronze_dev.global_mart_retail.raw_data"
silver_table_dim_product = "silver_dev.global_mart_retail.dim_product"

# ==========================================================
# Generate ONE batch_id for entire run
# ==========================================================
BATCH_ID = str(uuid.uuid4())

# ==========================================================
# 1. Read Bronze Data
# ==========================================================
bronze_df = spark.read.table(bronze_table)

# ==========================================================
# 2. Clean & Standardize Product Attributes
# ==========================================================
cleaned_df = (
    bronze_df
    .select(
        F.upper(F.trim(F.col("product_id"))).alias("product_id"),
        F.lower(F.trim(F.col("product_name"))).alias("product_name"),
        F.coalesce(F.lower(F.trim(F.col("category"))), F.lit("unknown")).alias("category"),
        F.coalesce(F.lower(F.trim(F.col("sub-category"))), F.lit("unknown")).alias("sub_category"),
        F.col("ingestion_ts"),
        F.col("row_id")
    )
)

# ==========================================================
# 3. Generate Business Hash (Change Detection)
# ==========================================================
hashed_df = (
    cleaned_df
    .withColumn(
        "product_hash",
        F.sha2(
            F.concat_ws(
                "||",
                "product_name",
                "category",
                "sub_category"
            ),
            256
        )
    )
)

# ==========================================================
# 4. HARD SOURCE DE-DUPLICATION
#    One row per (product_id + product_hash)
# ==========================================================
dedup_window = (
    Window
    .partitionBy("product_id", "product_hash")
    .orderBy(F.col("ingestion_ts").desc(), F.col("row_id").desc())
)

deduped_df = (
    hashed_df
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# ==========================================================
# 5. Assign is_current within THIS batch
# ==========================================================
current_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("ingestion_ts").desc(), F.col("row_id").desc())
)

staged_df = (
    deduped_df
    .withColumn("rn", F.row_number().over(current_window))
    .withColumn("is_current_record", F.col("rn") == 1)
    .withColumn("effective_start_timestamp", F.col("ingestion_ts"))
    .withColumn("effective_end_timestamp", F.lit(None).cast("timestamp"))
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("batch_id", F.lit(BATCH_ID))
    .drop("rn")
)

# ==========================================================
# 6. Create Silver Table if Not Exists
# ==========================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table_dim_product} (
    product_key BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING,
    product_name STRING,
    category STRING,
    sub_category STRING,
    product_hash STRING,
    effective_start_timestamp TIMESTAMP,
    effective_end_timestamp TIMESTAMP,
    is_current_record BOOLEAN,
    load_timestamp TIMESTAMP,
    batch_id STRING
)
USING DELTA
""")

silver_delta = DeltaTable.forName(spark, silver_table_dim_product)

# ==========================================================
# 7. MERGE — INSERT NEW PRODUCT HASHES ONLY
# ==========================================================
(
    silver_delta.alias("t")
    .merge(
        staged_df.alias("s"),
        "t.product_id = s.product_id AND t.product_hash = s.product_hash"
    )
    .whenNotMatchedInsert(
        values={
            "product_id": "s.product_id",
            "product_name": "s.product_name",
            "category": "s.category",
            "sub_category": "s.sub_category",
            "product_hash": "s.product_hash",
            "effective_start_timestamp": "s.effective_start_timestamp",
            "effective_end_timestamp": "s.effective_end_timestamp",
            "is_current_record": "s.is_current_record",
            "load_timestamp": "s.load_timestamp",
            "batch_id": "s.batch_id"
        }
    )
    .execute()
)

# ==========================================================
# 7A. Capture MERGE Metrics (INSERT COUNT)
# ==========================================================
history_df = silver_delta.history(1)

metrics = (
    history_df
    .select("operationMetrics")
    .collect()[0]["operationMetrics"]
)

inserted_rows = int(metrics.get("numTargetRowsInserted", 0))

# ==========================================================
# 8. FIX is_current & effective_end_timestamp (GLOBAL SCD2)
# ==========================================================
silver_df = spark.read.table(silver_table_dim_product)

scd_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("effective_start_timestamp").desc())
)

scd_updates = (
    silver_df
    .withColumn("rn", F.row_number().over(scd_window))
    .withColumn("new_is_current", F.col("rn") == 1)
    .withColumn(
        "new_effective_end_timestamp",
        F.when(F.col("rn") == 1, None)
         .otherwise(F.lag("effective_start_timestamp").over(scd_window))
    )
    .select(
        "product_key",
        "new_is_current",
        "new_effective_end_timestamp"
    )
)

(
    silver_delta.alias("t")
    .merge(
        scd_updates.alias("s"),
        "t.product_key = s.product_key"
    )
    .whenMatchedUpdate(
        set={
            "is_current_record": "s.new_is_current",
            "effective_end_timestamp": "s.new_effective_end_timestamp"
        }
    )
    .execute()
)

print("dim_product MERGE Metrics")
print(f"Records inserted in this run : {inserted_rows}")
print(f"batch_id                    : {BATCH_ID}")
print(f"dim_product SCD Type 2 load completed successfully | batch_id = {BATCH_ID}")


In [0]:

# ==========================================================
# 6. Reconciliation & Validation Checks
# ==========================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT product_id) AS distinct_products,
    SUM(CASE WHEN is_current_record THEN 1 ELSE 0 END) AS current_records
FROM {silver_table_dim_product}
""").show()

In [0]:
spark.sql(f"""
SELECT product_id, COUNT(*) AS versions
FROM {silver_table_dim_product}
GROUP BY product_id
HAVING COUNT(*) > 1
ORDER BY versions DESC
""").show()

In [0]:
%sql
select * from silver_dev.global_mart_retail.dim_product where product_id = 'OFF-PA-10002377'